## Librerias

In [18]:
import pandas as pd
from scipy.stats import ttest_ind
import statsmodels.formula.api as smf

## Carga datos clean

In [19]:
df_ops = pd.read_csv('C:\\Users\\Usuario\\ProjecteData\\Equip_25\\Data\\clean_data_16-03-2026.csv', parse_dates=['insert_date'])

# inventario activo
df_active = df_ops[df_ops["has_availability"] == True].copy()

df_active.info()
df_active.head()

<class 'pandas.core.frame.DataFrame'>
Index: 7159 entries, 0 to 7692
Data columns (total 37 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   apartment_id                 7159 non-null   int64         
 1   name                         7156 non-null   object        
 2   description                  7159 non-null   object        
 3   host_id                      7159 non-null   int64         
 4   neighbourhood_name           7159 non-null   object        
 5   neighbourhood_district       7159 non-null   object        
 6   room_type                    7159 non-null   object        
 7   accommodates                 7159 non-null   int64         
 8   bathrooms                    7117 non-null   float64       
 9   bedrooms                     7122 non-null   float64       
 10  beds                         7152 non-null   float64       
 11  amenities_list               7159 non-null   obj

,apartment_id,name,description,host_id,neighbourhood_name,neighbourhood_district,room_type,accommodates,bathrooms,bedrooms,...,review_scores_communication,review_scores_location,review_scores_value,is_instant_bookable,reviews_per_month,country,city,insert_date,price_missing,reviews 80+
0,11964,A ROOM WITH A VIEW,Private bedroom in our attic apartment. Right ...,45553,Centro,(sin contestar),Private room,2,2.0,1.0,...,100.0,100.0,100.0,False,75.0,spain,Malaga,2018-07-31,False,True
1,21853,Bright and airy room,We have a quiet and sunny room with a good vie...,83531,C�rmenes,Latina,Private room,1,1.0,1.0,...,100.0,80.0,90.0,False,52.0,spain,Madrid,2020-01-10,False,True
2,32347,Explore Cultural Sights from a Family-Friendly...,Open French doors and step onto a plant-filled...,139939,San Vicente,Casco Antiguo,Entire home/apt,4,1.0,2.0,...,100.0,100.0,100.0,True,142.0,spain,Sevilla,2019-07-29,False,True
3,35379,Double 02 CasanovaRooms Barcelona,Room at a my apartment. Kitchen and 2 bathroom...,152232,l'Antiga Esquerra de l'Eixample,Eixample,Private room,2,2.0,1.0,...,100.0,100.0,90.0,True,306.0,spain,Barcelona,2020-01-10,False,True
4,35801,Can Torras Farmhouse Studio Suite,Lay in bed & watch sunlight change the mood of...,153805,Quart,(sin contestar),Private room,5,1.0,2.0,...,100.0,100.0,100.0,False,39.0,spain,Girona,2019-02-19,False,True


In [20]:
host_counts = df_active.groupby("host_id")["apartment_id"].nunique()

host_counts.describe()
host_counts.value_counts().head(10)

apartment_id
1     4808
2      356
3      110
4       54
5       38
6       19
7       16
8       15
9        8
12       5
Name: count, dtype: int64

#### Definición de métricas

In [21]:
# ocupación
df_active["occ_30"] = (30 - df_active["availability_30"]) / 30
df_active["occ_60"] = (60 - df_active["availability_60"]) / 60
df_active["occ_90"] = (90 - df_active["availability_90"]) / 90
df_active["occ_365"] = (365 - df_active["availability_365"]) / 365

occupancy_cols = ["occ_30", "occ_60", "occ_90", "occ_365"]

# días ocupados 
df_active["occupied_days_30"] = 30 - df_active["availability_30"]   

monthly_occupancy_rate = round(
    (df_active["occupied_days_30"].sum() /
     (df_active.shape[0] * 30)) * 100,
    2
)

# ocupación promedio por horizonte temporal
occupancy_rates = (df_active[occupancy_cols].mean() * 100).round(2)   
df_active["is_instant_bookable"].value_counts()

is_instant_bookable
True     4093
False    3066
Name: count, dtype: int64

## Análisis / visualizaciones

#### Global
Ocupación media  (instant vs no)

In [22]:
global_comparison = (
    df_active
    .groupby("is_instant_bookable")["occ_30"]
    .agg(["mean", "median", "count"])
    .round(3)
)

global_comparison

,mean,median,count
is_instant_bookable,,,
False,0.577,0.667,3066
True,0.592,0.667,4093


In [23]:
global_comparison = (
    df_active
    .groupby("is_instant_bookable")["availability_30"]
    .agg(["mean", "median", "count"])
    .round(3)
)

global_comparison

,mean,median,count
is_instant_bookable,,,
False,12.704,10.0,3066
True,12.248,10.0,4093


A nivel global los alojamientos con instant booking presentan mayor ocupación media

#### Tamaño de muestra

In [24]:
city_room_comparison = (
    df_active
    .groupby(["city", "room_type", "is_instant_bookable"])["occ_30"]
    .mean()
    .unstack()
    .round(3)
)

# eliminar casos sin ambos grupos
city_room_comparison = city_room_comparison.dropna()

# calcular uplift
city_room_comparison["uplift"] = (
    (city_room_comparison[True] - city_room_comparison[False]) * 100
).round(1)

city_room_comparison = city_room_comparison.sort_values("uplift", ascending=False)
city_room_comparison

is_instant_bookable        False   True  uplift
city      room_type                            
Girona    Private room     0.367  0.556    18.9
          Entire home/apt  0.453  0.561    10.8
Malaga    Private room     0.544  0.645    10.1
Menorca   Entire home/apt  0.470  0.558     8.8
Madrid    Shared room      0.467  0.539     7.2
Barcelona Private room     0.604  0.650     4.6
Valencia  Private room     0.517  0.559     4.2
          Entire home/apt  0.560  0.601     4.1
Madrid    Private room     0.661  0.682     2.1
Barcelona Entire home/apt  0.611  0.616     0.5
Mallorca  Private room     0.492  0.487    -0.5
          Entire home/apt  0.577  0.557    -2.0
Sevilla   Entire home/apt  0.525  0.503    -2.2
Madrid    Entire home/apt  0.650  0.627    -2.3
Malaga    Entire home/apt  0.601  0.576    -2.5
Sevilla   Private room     0.623  0.583    -4.0
Barcelona Hotel room       0.800  0.606   -19.4
Menorca   Private room     0.744  0.538   -20.6
Barcelona Shared room      0.576  0.230   -34.6
Mallorca  Shared room      1.000  0.433   -56.7

“Instant booking no es una estrategia universalmente efectiva; su impacto depende del mercado.”
Impacto de instant booking positivo en Girona, moderado en Menorca y Valencia --> instant booking aumenta reservas / más relevante en mercados menos saturados

En Barcelona impacto casi nulo --> la demanda ya es alta, por lo que no necesita/afecta instant booking / otros factores dominan (ubicación, reviews, precios,etc)

En Madrid y Sevilla impacto negativo --> Sesgo de selección (listings de menor calidad usan instant booking) / mercado ya eficiente (fricción no es problema) / segmentación distinta (tipos de alojamientos diferentes)

#### Comparación por ciudad

In [25]:
sample_size = (
    df_active
    .groupby(["city", "is_instant_bookable"])["apartment_id"]
    .count()
    .unstack()
)

sample_size

is_instant_bookable,False,True
city,,
Barcelona,918,932
Girona,487,711
Madrid,673,887
Malaga,121,273
Mallorca,482,745
Menorca,78,89
Sevilla,129,274
Valencia,178,182


In [26]:
size_room = (
    df_active
    .groupby(["city", "room_type", "is_instant_bookable"])["apartment_id"]
    .count()
    .unstack()
)

# alinear índices
size_room = size_room.loc[city_room_comparison.index]

# filtrar muestras pequeñas
city_room_comparison_filtered = city_room_comparison[
    (size_room[True] >= 20) & (size_room[False] >= 20)
]
size_room

is_instant_bookable        False  True 
city      room_type                    
Girona    Private room      29.0   35.0
          Entire home/apt  458.0  670.0
Malaga    Private room      31.0   33.0
Menorca   Entire home/apt   75.0   81.0
Madrid    Shared room        9.0    6.0
Barcelona Private room     541.0  426.0
Valencia  Private room      48.0   36.0
          Entire home/apt  130.0  141.0
Madrid    Private room     303.0  236.0
Barcelona Entire home/apt  369.0  485.0
Mallorca  Private room      41.0   53.0
          Entire home/apt  440.0  680.0
Sevilla   Entire home/apt   88.0  221.0
Madrid    Entire home/apt  361.0  634.0
Malaga    Entire home/apt   90.0  234.0
Sevilla   Private room      40.0   44.0
Barcelona Hotel room         1.0   12.0
Menorca   Private room       3.0    8.0
Barcelona Shared room        7.0    9.0
Mallorca  Shared room        1.0    3.0

#### Test estadístico

In [27]:
model = smf.ols(
    "occ_30 ~ is_instant_bookable + C(room_type) + C(city)",
    data=df_active
).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                 occ_30   R-squared:                       0.016
Model:                            OLS   Adj. R-squared:                  0.015
Method:                 Least Squares   F-statistic:                     10.85
Date:                Wed, 18 Mar 2026   Prob (F-statistic):           3.80e-20
Time:                        13:21:57   Log-Likelihood:                -3295.9
No. Observations:                7159   AIC:                             6616.
Df Residuals:                    7147   BIC:                             6698.
Df Model:                          11                                         
Covariance Type:            nonrobust                                         
                                   coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------
Intercept       

## Resultados / archivos generados

In [49]:
df_base = df_active.copy()

In [43]:
room_type_dist = (
    df_active
    .groupby("room_type")
    .agg(
        listings=("apartment_id", "count")
    )
    .reset_index()
)

# total
total_listings = room_type_dist["listings"].sum()

# porcentaje
room_type_dist["share_%"] = (
    room_type_dist["listings"] / total_listings * 100
).round(1)

# ordenar de mayor a menor
room_type_dist = room_type_dist.sort_values("share_%", ascending=False)

room_type_dist

,room_type,listings,share_%
0,Entire home/apt,5157,72.0
2,Private room,1907,26.6
1,Hotel room,53,0.7
3,Shared room,42,0.6


#### Mejora por segmento

In [52]:
df_base = df_base[
    df_base["room_type"].isin(["Entire home/apt", "Private room"])
]
impact_segment = (
    df_base
    .groupby(["city", "room_type", "is_instant_bookable"])
    .agg(
        occupancy=("occ_30", "mean")
    )
    .reset_index()
)

# pivot para comparar
impact_pivot = impact_segment.pivot_table(
    index=["city", "room_type"],
    columns="is_instant_bookable",
    values="occupancy"
).reset_index()

# calcular diferencia
impact_pivot["difference_%"] = (
    (impact_pivot[True] - impact_pivot[False]) * 100
).round(1)

# crear nombre de segmento
impact_pivot["segment"] = (
    impact_pivot["city"] + " - " + impact_pivot["room_type"]
)

# ordenar (clave)
impact_pivot = impact_pivot.sort_values("difference_%", ascending=False)

# columnas finales
impact_final = impact_pivot[[
    "segment", "difference_%"
]]

impact_final.dropna(inplace=True)


# export
impact_final.to_csv(
    "C:\\Users\\Usuario\\Desktop\\PROYECTO\\04_impact_segment.csv",
    index=False
)

impact_final

C:\Users\Usuario\AppData\Local\Temp\ipykernel_29724\320873211.py:38: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  impact_final.dropna(inplace=True)


is_instant_bookable,segment,difference_%
3,Girona - Private room,19.0
2,Girona - Entire home/apt,10.9
7,Malaga - Private room,10.1
10,Menorca - Entire home/apt,8.7
1,Barcelona - Private room,4.6
15,Valencia - Private room,4.2
14,Valencia - Entire home/apt,4.1
5,Madrid - Private room,2.1
0,Barcelona - Entire home/apt,0.5
9,Mallorca - Private room,-0.4


In [ ]:
regression_summary = pd.DataFrame({
    "metric": ["instant_booking_effect"],
    "value": [model.params["is_instant_bookable[T.True]"]],
    "value_%": [model.params["is_instant_bookable[T.True]"] * 100],
    "p_value": [model.pvalues["is_instant_bookable[T.True]"]]
})

regression_summary.to_csv(
    "C:\\Users\\Usuario\\Desktop\\PROYECTO\\operaciones_regression_summary.csv",    
    index=False)